# 06 — Interactive Prediction Tool
## FMD Outbreak Prediction — Sri Lanka

**Objective:** This interactive notebook allows you to input custom weather
and environmental data for any district and month to see the model's
real-time outbreak prediction.

*Note: This is an excellent tool for demonstrating the model's capabilities
during a presentation or viva.*

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import ipywidgets as widgets
from IPython.display import display, HTML

# ── 1. Load Data & Models ─────────────────────────────────────────────────────
BASE_DIR  = r'..'
DATA_FILE = os.path.join(BASE_DIR, 'data', 'processed',
                         'FMD_model_ready_main refined_final_dataset.csv')
MODEL_DIR = os.path.join(BASE_DIR, 'models')

# Load the trained model and scaler
with open(os.path.join(MODEL_DIR, 'final_model_stage1.pkl'), 'rb') as f:
    model = pickle.load(f)
with open(os.path.join(MODEL_DIR, 'scaler_stage1.pkl'), 'rb') as f:
    scaler = pickle.load(f)

# Load dataset just to extract the static district mappings (lat, lon, densities)
df = pd.read_csv(DATA_FILE)

# Encode district as we did during training to get the mapping
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['district_enc'] = le.fit_transform(df['district'])

# Create a dictionary of static features for each district
district_static = {}
for dist in df['district'].unique():
    row = df[df['district'] == dist].iloc[0]
    district_static[dist] = {
        'lat': row['lat'],
        'lon': row['lon'],
        'buffalo_density': row['buffalo_density'],
        'livestock_density': row['livestock_density'],
        'district_enc': row['district_enc']
    }

print("✅ Models and static district data loaded successfully!")

✅ Models and static district data loaded successfully!


---
## 2. The Prediction Engine
This function takes the user inputs, calculates the complex features
(like sine/cosine of the month and monsoon phases), scales everything,
and outputs the final prediction.

In [2]:
def make_prediction(district, month, rainfall_mm, r3h, rfq, rain_lag1, rain_lag2, rfq_lag1,
                    humidity, wind_speed, temp_lag1, humidity_lag1, wind_lag1):
    
    # 1. Cyclical Month Encoding
    sin_month = np.sin(2 * np.pi * month / 12)
    cos_month = np.cos(2 * np.pi * month / 12)
    
    # 2. Monsoon Phase Encoding
    m_first = 1 if month in [3, 4] else 0
    m_sw = 1 if month in [5, 6, 7, 8, 9] else 0
    m_second = 1 if month in [10, 11] else 0
    m_ne = 1 if month in [12, 1, 2] else 0
    
    # 3. Retrieve District Static Data
    static = district_static[district]
    
    # 4. Construct Feature Vector (MUST match training order exactly)
    # The expected order from Notebook 05 is:
    # ['sin_month', 'cos_month', 'monsoon_phase_First_Inter_Monsoon', 'monsoon_phase_SW_Monsoon', 
    #  'monsoon_phase_Second_Inter_Monsoon', 'monsoon_phase_NE_Monsoon', 'rainfall_mm', 'r3h', 'rfq', 
    #  'rain_lag1', 'rain_lag2', 'rfq_lag1', 'lat', 'lon', 'humidity', 'wind_speed', 'temp_lag1', 
    #  'humidity_lag1', 'wind_lag1', 'buffalo_density', 'livestock_density', 'district_enc']
    
    features = np.array([[
        sin_month, cos_month, m_first, m_sw, m_second, m_ne,
        rainfall_mm, r3h, rfq, rain_lag1, rain_lag2, rfq_lag1,
        static['lat'], static['lon'], humidity, wind_speed, 
        temp_lag1, humidity_lag1, wind_lag1, 
        static['buffalo_density'], static['livestock_density'], static['district_enc']
    ]])
    
    # 5. Scale features
    features_scaled = scaler.transform(features)
    
    # 6. Predict
    prob = model.predict_proba(features_scaled)[0, 1]
    prediction = int(model.predict(features_scaled)[0])
    
    return prediction, prob

---
## 3. Interactive Prediction Dashboard
Run this cell to display the UI. Enter your values and click **Predict**.

In [3]:
# Create UI Widgets
style = {'description_width': 'initial'}

dist_w = widgets.Dropdown(options=sorted(list(district_static.keys())), description='District:', style=style)
month_w = widgets.IntSlider(min=1, max=12, value=1, description='Month (1-12):', style=style)

# Weather inputs (Current Month)
w_rain = widgets.FloatText(value=100.0, description='Rainfall (mm):', style=style)
w_r3h = widgets.FloatText(value=50.0, description='Max 3-Day Rain (r3h):', style=style)
w_rfq = widgets.FloatText(value=10.0, description='Rainy Days (rfq):', style=style)
w_hum = widgets.FloatText(value=75.0, description='Humidity (%):', style=style)
w_wind = widgets.FloatText(value=3.5, description='Wind Speed:', style=style)

# Weather inputs (Previous Months / Lags)
w_rain_l1 = widgets.FloatText(value=90.0, description='Rain Lag-1 (mm):', style=style)
w_rain_l2 = widgets.FloatText(value=80.0, description='Rain Lag-2 (mm):', style=style)
w_rfq_l1 = widgets.FloatText(value=8.0, description='Rainy Days Lag-1:', style=style)
w_temp_l1 = widgets.FloatText(value=28.0, description='Temp Lag-1 (°C):', style=style)
w_hum_l1 = widgets.FloatText(value=70.0, description='Humidity Lag-1 (%):', style=style)
w_wind_l1 = widgets.FloatText(value=3.0, description='Wind Lag-1:', style=style)

btn_predict = widgets.Button(description="Predict Outbreak", button_style='danger', 
                             icon='exclamation-triangle', layout=widgets.Layout(width='auto', height='40px'))
out = widgets.Output()

def on_button_clicked(b):
    with out:
        out.clear_output()
        pred, prob = make_prediction(
            dist_w.value, month_w.value, w_rain.value, w_r3h.value, w_rfq.value,
            w_rain_l1.value, w_rain_l2.value, w_rfq_l1.value,
            w_hum.value, w_wind.value, w_temp_l1.value, w_hum_l1.value, w_wind_l1.value
        )
        
        print("-" * 50)
        print(f"🌍 District : {dist_w.value}")
        print(f"📅 Month    : {month_w.value}")
        print("-" * 50)
        
        if pred == 1:
            display(HTML(f"<h2 style='color:red;'>🚨 HIGH RISK: Outbreak Predicted!</h2>"))
        else:
            display(HTML(f"<h2 style='color:green;'>✅ LOW RISK: No Outbreak Predicted</h2>"))
            
        print(f"Confidence (Probability of Outbreak): {prob*100:.2f}%\n")

btn_predict.on_click(on_button_clicked)

# Layout
ui_col1 = widgets.VBox([widgets.HTML("<b>Location & Time</b>"), dist_w, month_w, 
                        widgets.HTML("<br><b>Current Weather</b>"), w_rain, w_r3h, w_rfq, w_hum, w_wind])
ui_col2 = widgets.VBox([widgets.HTML("<br><b>Previous Month Weather (Lags)</b>"), 
                        w_rain_l1, w_rain_l2, w_rfq_l1, w_temp_l1, w_hum_l1, w_wind_l1])

dashboard = widgets.VBox([
    widgets.HBox([ui_col1, ui_col2]),
    widgets.HTML("<br>"),
    btn_predict,
    out
])

display(dashboard)